# TMDB Movie Recommender Data Cleaning

This notebook prepares a clean TMDB working dataset for the Flask hybrid recommender. The raw CSV in this repository contains more than one million rows, so the workflow uses a 10,000-row working slice to keep experimentation and app startup practical.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_PATH = Path('movies1M.csv')
OUTPUT_PATH = Path('artifacts/tmdb_movies_10k_clean.csv')
SAMPLE_SIZE = 10_000

USE_COLS = [
    'id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'runtime',
    'backdrop_path', 'original_language', 'overview', 'popularity', 'poster_path',
    'tagline', 'genres', 'production_companies', 'production_countries',
    'spoken_languages', 'keywords'
]


In [ ]:
df = pd.read_csv(RAW_PATH, usecols=USE_COLS, nrows=SAMPLE_SIZE, low_memory=False)
df.head(3)


In [ ]:
text_cols = [
    'title', 'overview', 'tagline', 'genres', 'production_companies',
    'production_countries', 'spoken_languages', 'keywords', 'original_language'
]
for col in text_cols:
    df[col] = df[col].fillna('').astype(str)

for col in ['vote_average', 'vote_count', 'popularity', 'runtime']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = df['release_date'].dt.year.fillna(0).astype(int)
df['id'] = pd.to_numeric(df['id'], errors='coerce')
df = df.dropna(subset=['id', 'title']).copy()
df['id'] = df['id'].astype(int)

def split_values(value: str) -> list[str]:
    return [item.strip() for item in value.split(',') if item and item.strip()]

df['genre_list'] = df['genres'].map(split_values)
df['keyword_list'] = df['keywords'].map(split_values)
df['spoken_language_list'] = df['spoken_languages'].map(split_values)
df['primary_language'] = df['original_language'].str.lower().str.strip().replace('', 'unknown')
df['normalized_title'] = df['title'].str.casefold().str.strip()

vote_mean = df['vote_average'].mean()
vote_threshold = df['vote_count'].quantile(0.75)
df['imdb_rating'] = df.apply(
    lambda row: (row['vote_count'] / (row['vote_count'] + vote_threshold)) * row['vote_average']
    + (vote_threshold / (row['vote_count'] + vote_threshold)) * vote_mean
    if row['vote_count'] > 0 else vote_mean,
    axis=1,
)

def minmax(series: pd.Series) -> pd.Series:
    minimum = series.min()
    maximum = series.max()
    if maximum == minimum:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - minimum) / (maximum - minimum)

df['popularity_score'] = minmax(df['popularity'])
df['imdb_score_norm'] = minmax(df['imdb_rating'])
df['vote_score_norm'] = minmax(df['vote_count'])
df['discovery_score'] = (
    0.55 * df['imdb_score_norm'] + 0.25 * df['popularity_score'] + 0.20 * df['vote_score_norm']
)

def build_content_soup(row: pd.Series) -> str:
    weighted_bits = []
    weighted_bits.extend(row['genre_list'] * 3)
    weighted_bits.extend(row['keyword_list'] * 2)
    weighted_bits.extend(split_values(row['production_companies']))
    weighted_bits.extend(split_values(row['production_countries']))
    weighted_bits.extend(split_values(row['spoken_languages']))
    weighted_bits.extend([row['primary_language']] * 2)
    weighted_bits.append(row['overview'])
    weighted_bits.append(row['tagline'])
    return ' '.join(bit for bit in weighted_bits if bit)

df['content_soup'] = df.apply(build_content_soup, axis=1)
df = df.drop_duplicates(subset=['normalized_title'], keep='first').reset_index(drop=True)
df.shape


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved cleaned dataset to {OUTPUT_PATH.resolve()}')
df[['title', 'genres', 'primary_language', 'imdb_rating', 'discovery_score']].head(10)
